<a href="https://colab.research.google.com/github/agarwalpratik/aiml/blob/main/Automating_Port_Operations_DeepLearning_Part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

2.	Build a lightweight model with the aim of deploying the solution on a mobile device using transfer learning. You can use any lightweight pre-trained model as the initial (first) layer. MobileNetV2 is a popular lightweight pre-trained model built using Keras API.

2.1.	Split the dataset into train and test datasets in the ration 70:30, with shuffle and random state=1.

2.2.	Use tf.keras.preprocessing.image_dataset_from_directory to load the train and test datasets. This function also supports data normalization.
(Hint: Image_scale=1./255).

2.3.	Load train, validation and test datasets in batches of 32 using the function initialized in the above step.

2.4.	Build a CNN network using Keras with the following layers.

•	Load MobileNetV2 - Light Model as the first layer
(Hint: Keras API Doc)

•	GLobalAveragePooling2D layer

•	Dropout(0.2)

•	Dense layer with 256 neurons and activation relu

•	BatchNormalization layer

•	Dropout(0.1)

•	Dense layer with 128 neurons and activation relu

•	BatchNormalization layer

•	Dropout(0.1)

•	Dense layer with 9 neurons and activation softmax

2.5.	Compile the model with Adam optimizer, categorical_crossentropy loss, and metrics accuracy, Precision, and Recall.

2.6.	Train the model for 50 epochs and Early stopping while monitoring validation loss.

2.7.	Evaluate the model on test images and print the test loss and accuracy.

2.8.	Plot Train loss Vs Validation loss and Train accuracy Vs Validation accuracy.


In [4]:
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import os

tf.__version__

'2.17.1'

In [2]:
#Extract the Zip File

!unzip boat_type_classification_dataset.zip


Archive:  boat_type_classification_dataset.zip
   creating: boat_type_classification_dataset/
   creating: boat_type_classification_dataset/buoy/
  inflating: boat_type_classification_dataset/buoy/1.jpg  
  inflating: boat_type_classification_dataset/buoy/10.jpg  
  inflating: boat_type_classification_dataset/buoy/11.jpg  
  inflating: boat_type_classification_dataset/buoy/12.jpg  
  inflating: boat_type_classification_dataset/buoy/13.jpg  
  inflating: boat_type_classification_dataset/buoy/14.jpg  
  inflating: boat_type_classification_dataset/buoy/15.jpg  
  inflating: boat_type_classification_dataset/buoy/16.jpg  
  inflating: boat_type_classification_dataset/buoy/17.jpg  
  inflating: boat_type_classification_dataset/buoy/18.jpg  
  inflating: boat_type_classification_dataset/buoy/19.jpg  
  inflating: boat_type_classification_dataset/buoy/2.jpg  
  inflating: boat_type_classification_dataset/buoy/20.jpg  
  inflating: boat_type_classification_dataset/buoy/21.jpg  
  inflating: boa

In [3]:
#2.1.	Split the dataset into train and test datasets in the ration 70:30, with shuffle and random state=1.
#2.2.	Use tf.keras.preprocessing.image_dataset_from_directory to load the train and test datasets. This function also supports data normalization.
#(Hint: Image_scale=1./255).
#2.3.	Load train, validation and test datasets in batches of 32 using the function initialized in the above step.

my_generator = tf.keras.preprocessing.image.ImageDataGenerator(rescale=1.0/255.0,
                                                               validation_split=0.3)

trainImageData = my_generator.flow_from_directory('boat_type_classification_dataset',
                                                     batch_size=32,
                                                     class_mode='categorical',
                                                     seed=1,
                                                     target_size=(128,128),
                                                     subset='training')

testImageData = my_generator.flow_from_directory('boat_type_classification_dataset',
                                                     batch_size=32,
                                                     class_mode='categorical',
                                                     seed=1,
                                                     target_size=(128,128),
                                                     subset='validation')


Found 820 images belonging to 9 classes.
Found 342 images belonging to 9 classes.


2.4.	Build a CNN network using Keras with the following layers.

•	Load MobileNetV2 - Light Model as the first layer
(Hint: Keras API Doc)

•	GLobalAveragePooling2D layer

•	Dropout(0.2)

•	Dense layer with 256 neurons and activation relu

•	BatchNormalization layer

•	Dropout(0.1)

•	Dense layer with 128 neurons and activation relu

•	BatchNormalization layer

•	Dropout(0.1)

•	Dense layer with 9 neurons and activation softmax


In [5]:
base_model = MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
base_model.trainable = False

model = tf.keras.Sequential()

model.add(base_model)

model.add(tf.keras.layers.GlobalAveragePooling2D())

model.add(tf.keras.layers.Dropout(0.2))

model.add(tf.keras.layers.Dense(units=256, activation='relu'))

model.add(tf.keras.layers.BatchNormalization())

model.add(tf.keras.layers.Dropout(0.1))

model.add(tf.keras.layers.Dense(units=128, activation='relu'))

model.add(tf.keras.layers.BatchNormalization())

model.add(tf.keras.layers.Dropout(0.1))

model.add(tf.keras.layers.Dense(units=9, activation='softmax'))


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


In [6]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_128 (Functional)    │ (None, 4, 4, 1280)          │       2,257,984 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 1280)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 256)                 │         327,936 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 256)                 │           1,024 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 256)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 128)                 │          32,896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 128)                 │             512 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 9)                   │           1,161 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 2,621,513 (10.00 MB)

 Trainable params: 362,761 (1.38 MB)

 Non-trainable params: 2,258,752 (8.62 MB)

In [7]:
#2.5.	Compile the model with Adam optimizer, categorical_crossentropy loss, and metrics accuracy, Precision, and Recall.

model.compile(optimizer="adam",
              loss="categorical_crossentropy",
              metrics=['accuracy', tf.keras.metrics.Precision(), tf.keras.metrics.Recall()])

In [ ]:
#2.6.	Train the model for 50 epochs and Early stopping while monitoring validation loss.

early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history = model.fit(
    trainImageData,
    validation_data=testImageData,
    epochs=50,
    callbacks=[early_stopping]
)

Epoch 1/50


/usr/local/lib/python3.10/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:122: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


26/26 ━━━━━━━━━━━━━━━━━━━━ 40s 928ms/step - accuracy: 0.3795 - loss: 1.9376 - precision: 0.5591 - recall: 0.2851 - val_accuracy: 0.7953 - val_loss: 0.7007 - val_precision: 0.9157 - val_recall: 0.6667
Epoch 2/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 9s 300ms/step - accuracy: 0.8523 - loss: 0.4764 - precision: 0.9042 - recall: 0.7798 - val_accuracy: 0.8246 - val_loss: 0.5938 - val_precision: 0.9039 - val_recall: 0.7427
Epoch 3/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 9s 257ms/step - accuracy: 0.9289 - loss: 0.2594 - precision: 0.9572 - recall: 0.8722 - val_accuracy: 0.8450 - val_loss: 0.5757 - val_precision: 0.9085 - val_recall: 0.7544
Epoch 4/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 10s 249ms/step - accuracy: 0.9368 - loss: 0.1968 - precision: 0.9667 - recall: 0.9053 - val_accuracy: 0.8333 - val_loss: 0.5336 - val_precision: 0.8914 - val_recall: 0.7924
Epoch 5/50
26/26 ━━━━━━━━━━━━━━━━━━━━ 9s 304ms/step - accuracy: 0.9604 - loss: 0.1497 - precision: 0.9759 - recall: 0.9411 - val_accuracy: 0.8333 - val_loss: 0.5581 - va